<a href="https://colab.research.google.com/github/eniompw/TinyLM/blob/main/SimpleTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset
import warnings, itertools
warnings.filterwarnings('ignore')

def load_tinystories(num_stories=200, context_size=4):
    """
    Fetches the TinyStories dataset and prepares it for a character-level language model.
    """
    # Fetch data
    dataset = load_dataset('karpathy/tinystories-gpt4-clean', split='train', streaming=True)
    text = ''.join(s['text'] for s in itertools.islice(dataset, num_stories))

    # Build vocabulary and tokenize
    vocab = sorted(set(text))                                       # ordered list of unique characters
    char_to_id = {c: i for i, c in enumerate(vocab)}                # dictionary mapping char to integer id
    encoded = [char_to_id[c] for c in text]                         # map entire text to integer sequence

    # If context_size=1, skip building windows to save RAM (training loop handles slicing).
    if context_size == 1:
        return [], [], vocab, encoded

    # Create sliding windows for inputs and targets using standard Python lists
    inputs = [encoded[i:i+context_size] for i in range(len(encoded)-context_size)] # sliding windows
    targets = encoded[context_size:]                                               # next char to predict

    return inputs, targets, vocab, encoded

In [6]:
import time, torch
import torch.nn as nn, torch.nn.functional as F
#from tinystories_dataset import load_tinystories

# Automatically create all tensors on GPU if available, removing manual device boilerplate
torch.set_default_device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Hyperparameters ---
context_size = 8                                                                                  # number of previous tokens used to predict next
embed_dim    = 128                                                                                # token/positional embedding dimension
n_heads      = 4                                                                                  # number of attention heads in each transformer layer
ffn_dim      = 256                                                                                # feed-forward network hidden dimension
n_layers     = 2                                                                                  # number of transformer encoder layers
batch_size   = 1024                                                                               # number of samples per training step
lr           = 1e-3                                                                               # learning rate
n_steps      = 2001                                                                               # total training steps

# --- Data & Tokenization ---
input_ids, target_ids, idx_to_char, token_ids = load_tinystories(num_stories=200, context_size=context_size)
input_ids, target_ids = torch.tensor(input_ids), torch.tensor(target_ids)                        # convert to tensors

# --- Model ---
torch.manual_seed(42)                                                                             # seed helper for reproducibility
tok_embed   = nn.Embedding(len(idx_to_char), embed_dim)                                          # token embedding lookup layer
pos_embed   = nn.Embedding(context_size, embed_dim)                                              # positional embedding for sequence order
transformer = torch.compile(nn.TransformerEncoder(nn.TransformerEncoderLayer(embed_dim, n_heads, ffn_dim, batch_first=True, dropout=0.), n_layers))
linear      = nn.Linear(embed_dim, len(idx_to_char))                                             # maps hidden state to logits (vocab length)

params    = list(tok_embed.parameters()) + list(pos_embed.parameters()) + list(transformer.parameters()) + list(linear.parameters())
print(f"params: {sum(p.numel() for p in params):,}")
optimizer = torch.optim.Adam(params, lr=lr)                                                      # optimizer replaces manual parameter updates

# --- Train ---
start = time.time()                                                                               # track training duration
for step in range(n_steps):
    batch_idx = torch.randint(0, len(input_ids), (batch_size,))                                  # random batch indices
    x = tok_embed(input_ids[batch_idx]) + pos_embed(torch.arange(context_size))                 # add token and positional embeddings
    loss = F.cross_entropy(linear(transformer(x)[:, -1, :]), target_ids[batch_idx])              # computes softmax and cross-entropy loss automatically
    optimizer.zero_grad(); loss.backward(); optimizer.step()                                      # zero grads, backprop, Adam update

    if step % 200 == 0:
        with torch.no_grad():                                                                     # disable tracking during evaluation
            x_eval = tok_embed(input_ids) + pos_embed(torch.arange(context_size))               # embed full dataset
            pred_ids = linear(transformer(x_eval)[:, -1, :]).argmax(1)                          # full dataset forward & argmax
            print(f"Step {step:4d} | Loss: {loss:.4f} | Acc: {(pred_ids == target_ids).float().mean():.1%} | {time.time()-start:.1f}s")

print(f"Training time: {time.time() - start:.1f}s")

# --- Generate ---
@torch.no_grad()                                                                                  # disable autograd tracking during inference
def generate(num_chars=200, context_ids=list(token_ids[:context_size])):                         # start with true initial context
    output_chars = [idx_to_char[i] for i in context_ids]                                         # decode initial context to string
    for _ in range(num_chars):
        x = tok_embed(torch.tensor([context_ids])) + pos_embed(torch.arange(context_size))      # embed current window
        next_token_probs = torch.softmax(linear(transformer(x)[:, -1, :]) / 0.7, 1)             # apply temp 0.7 to pick higher-confidence tokens
        next_token = torch.multinomial(next_token_probs, 1).item()                               # randomly sample from predicted distribution
        context_ids, output_chars = context_ids[1:] + [next_token], output_chars + [idx_to_char[next_token]]  # slide window and append
    return ''.join(output_chars)

print(generate())

Step    0 | Loss: 4.4077 | Acc: 4.0% | 0.7s
Step  200 | Loss: 1.6047 | Acc: 53.5% | 4.1s
Step  400 | Loss: 1.3882 | Acc: 58.5% | 7.6s
Step  600 | Loss: 1.3462 | Acc: 60.7% | 11.0s
Step  800 | Loss: 1.2678 | Acc: 62.3% | 14.5s
Step 1000 | Loss: 1.1527 | Acc: 63.4% | 17.9s
Step 1200 | Loss: 1.1451 | Acc: 64.5% | 21.4s
Step 1400 | Loss: 1.1254 | Acc: 65.7% | 24.9s
Step 1600 | Loss: 1.1157 | Acc: 66.2% | 28.5s
Step 1800 | Loss: 1.1047 | Acc: 66.8% | 32.0s
Step 2000 | Loss: 1.0191 | Acc: 67.3% | 35.5s
Training time: 35.5s
Once there was a friends.
Tim was reminder your her the could be felt some man, but he was house saft. They looked up and closet he would fun were carron itside the sighter. They punknew that he could needive
